In [1]:
import sys
import os
sys.path.append("../")
sys.path.append("../../")

In [2]:
from scale_rl.common.wandb_utils import *

In [3]:
from scale_rl.envs.mujoco import MUJOCO_ALL, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE

/home/nas4_user/youngdolee/anaconda3/envs/rl_playground/lib/python3.9/site-packages/glfw/__init__.py:916: GLFWError: (65544) b'X11: The DISPLAY environment variable is missing'
  warnings.warn(message, GLFWError)


In [21]:
entity = 'draftrec'
project_name = 'crossq'

run_exp_names_to_analysis_exp_names = {}
_v = 'crossq'
for _env_name in MUJOCO_ALL:
    if _env_name == "Humanoid-v4":
        continue
    for _seed in [s * 1000 for s in range(10)]:
        _k = _env_name + f"_seed={_seed}"
        print(_k)
        run_exp_names_to_analysis_exp_names[_k] = _v
run_exp_names = list(run_exp_names_to_analysis_exp_names.keys())
metrics = ['eval/mean_reward']

HalfCheetah-v4_seed=0
HalfCheetah-v4_seed=1000
HalfCheetah-v4_seed=2000
HalfCheetah-v4_seed=3000
HalfCheetah-v4_seed=4000
HalfCheetah-v4_seed=5000
HalfCheetah-v4_seed=6000
HalfCheetah-v4_seed=7000
HalfCheetah-v4_seed=8000
HalfCheetah-v4_seed=9000
Hopper-v4_seed=0
Hopper-v4_seed=1000
Hopper-v4_seed=2000
Hopper-v4_seed=3000
Hopper-v4_seed=4000
Hopper-v4_seed=5000
Hopper-v4_seed=6000
Hopper-v4_seed=7000
Hopper-v4_seed=8000
Hopper-v4_seed=9000
Walker2d-v4_seed=0
Walker2d-v4_seed=1000
Walker2d-v4_seed=2000
Walker2d-v4_seed=3000
Walker2d-v4_seed=4000
Walker2d-v4_seed=5000
Walker2d-v4_seed=6000
Walker2d-v4_seed=7000
Walker2d-v4_seed=8000
Walker2d-v4_seed=9000
Ant-v4_seed=0
Ant-v4_seed=1000
Ant-v4_seed=2000
Ant-v4_seed=3000
Ant-v4_seed=4000
Ant-v4_seed=5000
Ant-v4_seed=6000
Ant-v4_seed=7000
Ant-v4_seed=8000
Ant-v4_seed=9000


In [22]:
runs = collect_runs(entity=entity, project_name=project_name)
for run in tqdm(runs):
    run.config["group_name"] = 'CrossQ'
    run.config['seed'] = int(run.config['seed'])
    run.config["env_name"] = run.config['env']
    run.config["exp_name"] = run.config['env'] + f"_seed={run.config['seed']}"

 
filtered_runs = filter_runs(runs, exp_names = run_exp_names)
wandb_df = convert_runs_to_dataframe(
    runs = filtered_runs, 
    run_exp_name_to_analysis_exp_name=run_exp_names_to_analysis_exp_names
)
wandb_df = wandb_df[wandb_df.apply(lambda row: 'finished' in str(row['run']), axis=1)]
eval_df = convert_wandb_df_to_eval_df(wandb_df, metrics)
eval_df['env_name'] = eval_df['env_name'].str.replace('_', '-')
eval_df


  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

,exp_name,env_name,seed,metric,env_step,value
0,crossq,Walker2d-v4,9000,eval/mean_reward,0,-13.216187
1,crossq,Walker2d-v4,9000,eval/mean_reward,87,37.418068
2,crossq,Walker2d-v4,9000,eval/mean_reward,107,1071.500977
3,crossq,Walker2d-v4,9000,eval/mean_reward,121,262.698578
4,crossq,Walker2d-v4,9000,eval/mean_reward,131,280.809631
...,...,...,...,...,...,...
2435,crossq,Ant-v4,0,eval/mean_reward,317,6974.143555
2436,crossq,Ant-v4,0,eval/mean_reward,322,7055.833496
2437,crossq,Ant-v4,0,eval/mean_reward,327,7087.139160
2438,crossq,Ant-v4,0,eval/mean_reward,332,7113.029297


In [17]:
def update_seeds_without_value_check(eval_df):
    """
    Updates the seed values in eval_df when exp_name, env_name, and env_step are the same.

    Args:
    - eval_df (pandas DataFrame): DataFrame containing the evaluation data.

    Returns:
    - eval_df (pandas DataFrame): Updated DataFrame with non-duplicate seed values for 
                                  matching exp_name, env_name, and env_step.
    """
    # Create a copy of the DataFrame to avoid modifying the original
    updated_df = eval_df.copy()

    # Track the seeds that have been used for each (exp_name, env_name, env_step)
    seed_tracker = {}

    # Iterate over the DataFrame row by row
    for idx, row in updated_df.iterrows():
        exp_name = row["exp_name"]
        env_name = row["env_name"]
        env_step = row["env_step"]
        metric = row['metric']
        seed = row["seed"]

        key = (exp_name, env_name, env_step, metric)

        if key not in seed_tracker:
            seed_tracker[key] = set()

        # If the current seed has already been used for this key (exp_name, env_name, env_step)
        if seed in seed_tracker[key]:
            # Increment the seed until we find a unique one
            new_seed = seed + 500
            while new_seed in seed_tracker[key]:
                new_seed += 500
            # Update the seed in the DataFrame
            updated_df.at[idx, "seed"] = new_seed
            # Add the new seed to the tracker
            seed_tracker[key].add(new_seed)
        else:
            # Add the current seed to the tracker
            seed_tracker[key].add(seed)

    return updated_df

In [18]:
eval_df = pd.concat([eval_df, sac_simba_eval_df], ignore_index=True)
new_df = update_seeds_without_value_check(eval_df)

In [19]:
new_df.to_csv("/home/nas4_user/youngdolee/SimbaV2/results/1219_sac_simba_1m.csv")